# Further Synchronisation

---

## Thread Safety

Thread-safe functions help us, as the end user, run a set of code without having to worry about synchronisation between threads.

As an example, `strtok` to handle string tokenisation is not thread-safe. This is because it makes use of a static internal variable, which is shared by all function calls on seperate threads. The reentrant version `strtok_r` fixes this by forcing the user to explicitly pass a pointer to its own state variable, so that there is no overlap between multiple threads.

In [2]:
%man strtok | head -n26

strtok(3)		    Library Functions Manual		      strtok(3)

NAME
       strtok, strtok_r - extract tokens from strings

LIBRARY
       Standard C library (libc, -lc)

SYNOPSIS
       #include <string.h>

       char *strtok(char *_Nullable restrict str, const char *restrict delim);
       char *strtok_r(char *_Nullable restrict str, const char *restrict delim,
		      char **restrict saveptr);

   Feature Test Macro Requirements for glibc (see feature_test_macros(7)):

       strtok_r():
	   _POSIX_C_SOURCE
	       || /* glibc <= 2.19: */ _BSD_SOURCE || _SVID_SOURCE

DESCRIPTION
       The  strtok()  function	breaks a string into a sequence of zero or more
       nonempty tokens.	 On the first call to strtok(), the string to be parsed
       should be specified in str.  In each subsequent call that  should  parse
       the same string, str must be NULL.


---

## Barriers

Barriers are a synchronisation point, which will block arriving threads until a certain number are ready to continue.

An example of a scenario that uses barriers could be a media player that needs most of its threads to have buffered data before starting playback. Another use case could be synchronising graphics and sound processing in a videogame, so that game logic stays consitent on each frame that is rendered. 

---

## Condition Variables

A condition variable is a synchronisation mechanism which lets an arbitrary number of threads sleep, to be woken up later by another thread. They are often used along with conditional checks.

The following requires a `mutex` to protect the actual variable itself.
```c
pthread_cond_wait(pthread_cond_t* restrict cond, pthread_mutex_t* restrict mutex);
```
The wait function forces the calling thread to sleep. Before the calling thread is blocked however, the `mutex` that it had access to is released, allowing another thread that was blocked to gain access.
```c
pthread_cond_signal(pthread_cond_t* cond);
```
This function randomly signals to wake one of the threads blocked on the condition variable. On being woken up thread regains access of the `mutex`.
```c
pthread_cond_broadcast(pthread_cond_t* cond);
```
This function signals to all threads blocked on the condition variable to be awoken.

---
---

# Scalable Algorithms

---

## Recursion

Recursion occurs when a procedure calls itself to solve a simpler version of the problem. In recursion we usually have a base case in which the problem can no longer be broken down, and the recursive step that defines how the problem gets broken down.

In C, every function call pushes a new stack frame onto the program stack. The stack isn't big enough enough for large amounts of recursive calls however, so if too many frames get pushed on, the stack will overflow and a segmentation fault will occur. 

If a recursive algorithm runs too slowly because of the overhead for procedure calls, it can always be implemented iteratively instead. (Tail recursion and related compiler optimisations can help).

---

## Divide and Conquer

A divide and conquer algorithm breaks the problem into several subproblems, solves the subproblems recursively, and combines to create a solution.
1. Divide the problem into a number of subproblems
2. Conquer subproblems directly if it is a base case, recursively otherwise
3. Combine subproblem solutions into solution for original problem

### Mergesort

1. Divide array into two halves
2. Conquer: recursively sort each half (until we reach 1-element base case)
3. Combine: merge two halves to make sorted whole (keep track of smallest element in each sorted half, insert smallest of two elements into array, repeat until done)

With unlimited parallelism, we generate new threads with every dividing step, down to the base case. In practice, thread creation comes with overhead, and there is a physical limit to the number of threads we can have.

With fixed parallelism, we generate new threads only until we reach a fixed limit. After that, we continue sequentially.

With scalable parallelism, the limit until we switch to sequential depends on the number of available cores.

---

## Reductions

A reduction operation reduces a collection of data items to a single data item by repeatedly combining the data items pairwise with a binary operator (e.g. $+, *, \max$).

Examples of reduction operations include sum of array, smallest array element, first index of element, etc.

So, if we are using threads to calculate the sum of an array, rather than all working on the same variable, have the seperate threads produce intermediate sums, and sum them up as the result.